# Fire / Smoke YOLO Training on Colab

This notebook mirrors your local `jetsonorin` project and runs training on Colab.

Steps:
- Clone your repo (or upload the folder).
- Install dependencies.
- Download and clean the dataset.
- Train YOLO with early stopping.
- Optionally export / benchmark.

**Before you run:**
- Replace `YOUR_REPO_URL_HERE` with your actual GitHub repo URL, or skip the clone and upload the project manually into `/content/jetsonorin`.


In [ ]:
#@title Clone repo or verify project directory
import os, shutil

project_dir = "/content/jetsonorin"  # change only if you use another folder name
repo_url = "YOUR_REPO_URL_HERE"      # e.g. "https://github.com/you/jetsonorin.git"

if repo_url != "YOUR_REPO_URL_HERE":
    # Clone fresh copy from GitHub
    if os.path.exists(project_dir):
        shutil.rmtree(project_dir)
    !git clone "$repo_url" "$project_dir"
else:
    # No repo URL: expect the project to already exist at project_dir
    if not os.path.exists(project_dir):
        raise FileNotFoundError(
            f"Project directory {project_dir} does not exist.\n"
            "Either set repo_url to your GitHub repo so Colab can clone it,\n"
            "or upload/unzip your project so it lives at that path."
        )
    print("No repo URL set. Using existing directory:", project_dir)

os.chdir(project_dir)
print("Working directory:", os.getcwd())
print("Project files:")
!ls -R


## Save best weights to Google Drive (optional)

Use this section if you want Colab to copy the trained `best.pt` into your Google Drive
so you can easily download it or sync it to your Jetson later.


In [ ]:
#@title Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")


In [ ]:
#@title Copy best.pt to Google Drive
from pathlib import Path
import shutil

MODEL_NAME = "yolov9"  #@param ["yolov8", "yolov9", "yolo11", "yolo12"]
drive_subdir = "MyDrive/jetsonorin_weights"  #@param {type:"string"}

project_dir = Path("fire_detection_runs_v2")
weights_path = project_dir / f"{MODEL_NAME}_fire" / "weights" / "best.pt"
if not weights_path.exists():
    raise FileNotFoundError(f"Could not find best weights at {weights_path}. Make sure training finished.")

drive_target_dir = Path("/content/drive") / drive_subdir
drive_target_dir.mkdir(parents=True, exist_ok=True)
target_path = drive_target_dir / f"{MODEL_NAME}_fire_best.pt"

shutil.copy2(weights_path, target_path)
print("Copied", weights_path, "to", target_path)


In [ ]:
#@title Install Python dependencies
# If you use a specific ultralytics version locally, keep it in sync here.
%%bash
pip install -U pip
pip install "ultralytics<9" kagglehub opencv-python pyyaml \
  polars seaborn matplotlib pandas requests tqdm psutil


## Configure Kaggle API (for dataset download)

You need a Kaggle API token. In Colab, the simplest options are:

- Upload your `.env` containing `KAGGLE_API_TOKEN=...` into the project root (next cell does that), **or**
- Set the environment variable directly in a cell.


In [ ]:
#@title Option 1: Set KAGGLE_API_TOKEN directly (recommended on Colab)
import os

KAGGLE_API_TOKEN = ""  #@param {type:"string"}
if KAGGLE_API_TOKEN:
    os.environ["KAGGLE_API_TOKEN"] = KAGGLE_API_TOKEN
    print("KAGGLE_API_TOKEN set in environment.")
else:
    print("No KAGGLE_API_TOKEN provided in this cell.")


In [ ]:
#@title Option 2: Use an uploaded .env file (optional)
from pathlib import Path
import os

env_path = Path(".env")
if env_path.exists():
    with env_path.open() as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ[key.strip()] = value.strip()
                print(f"Loaded {key.strip()} from .env")
else:
    print(".env not found in project root. You can upload it via the Colab file browser.")


In [ ]:
#@title Download dataset with download_dataset.py
%%bash
set -euo pipefail
cd /content/jetsonorin
python download_dataset.py


In [ ]:
#@title Ensure data.yaml paths are correct for Colab
from pathlib import Path
import yaml

data_yaml_path = Path("datasets/fire_smoke/data.yaml")
if data_yaml_path.exists():
    with data_yaml_path.open() as f:
        data_cfg = yaml.safe_load(f)
else:
    # If not present, fall back to top-level data.yaml from repo, if any
    top = Path("data.yaml")
    if top.exists():
        with top.open() as f:
            data_cfg = yaml.safe_load(f)
        data_yaml_path = top
    else:
        raise FileNotFoundError("Could not find datasets/fire_smoke/data.yaml or top-level data.yaml")

# Force paths to work on Colab under /content/jetsonorin
data_cfg["path"] = "datasets/fire_smoke"
data_cfg["train"] = "data/train/images"
data_cfg["val"] = "data/val/images"
data_cfg["test"] = "data/test/images"

with data_yaml_path.open("w") as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)

print("Updated", data_yaml_path, "to use relative paths under datasets/fire_smoke")


In [ ]:
#@title Preprocess / clean dataset
%%bash
set -euo pipefail
cd /content/jetsonorin
python preprocess_data.py --data datasets/fire_smoke/data.yaml


## Train YOLO (with early stopping)

This uses your existing `train_yolo.py`, which now supports:
- `--device auto|cpu|mps|0`
- `--patience` for early stopping

On Colab GPU we typically want:
- `device=0`
- a higher batch size (e.g. 8–16, depending on VRAM)


In [ ]:
#@title Run training
MODEL = "yolov9"   #@param ["yolov8", "yolov9", "yolo11", "yolo12"]
EPOCHS = 60        #@param {type:"integer"}
BATCH = 8          #@param {type:"integer"}
IMG_SIZE = 640     #@param {type:"integer"}
PATIENCE = 20      #@param {type:"integer"}
DEVICE = "0"      #@param ["0", "cpu"]

import subprocess, shlex

cmd = f"python train_yolo.py --models {MODEL} --data datasets/fire_smoke/data.yaml " \
      f"--epochs {EPOCHS} --batch {BATCH} --imgsz {IMG_SIZE} --patience {PATIENCE} --device {DEVICE}"

print("Running:", cmd)
result = subprocess.run(shlex.split(cmd), check=False)
print("Exit code:", result.returncode)


## (Optional) Export best weights to TensorRT-compatible formats

Colab can export to ONNX or engine formats, but these large files are usually
downloaded and then converted on the Jetson. Here we just call your existing
`export_models.py` if you want to try it.


In [ ]:
#@title Optional: export trained model(s)
from pathlib import Path

# Adjust these paths based on your chosen MODEL name and project/name in train_yolo.py
MODEL_NAME = "yolov9"  #@param ["yolov8", "yolov9", "yolo11", "yolo12"]
project_dir = Path("fire_detection_runs_v2")
weights_path = project_dir / f"{MODEL_NAME}_fire" / "weights" / "best.pt"

if not weights_path.exists():
    raise FileNotFoundError(f"Could not find best weights at {weights_path}. Run training first and check paths.")

print("Exporting", weights_path)
!python export_models.py --models "$weights_path"
